[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_chembl_precedent.ipynb)

# How much chemistry already exists for each target

**Orange group · Tuberculosis**

A target with five thousand published inhibitors is a very different project from a
target nobody has ever made a compound for. This notebook measures that for each of our
348 candidates, using ChEMBL, the public database of molecules that have been tested
against proteins: how many compounds were tested on the target itself, and how many on
its relatives in other organisms, whose chemistry is often a starting point.

## What you will do

- Load the 348 targets and their amino acid sequences.
- Ask ChEMBL what is known about one of them.
- Search the 11,000 proteins in ChEMBL for relatives of every target.
- Sort each relative into an exact match, a close homolog or a distant one.
- Count the unique compounds behind each target and save the table.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the targets and their sequences

We start from the shortlist made in `orange_target_selection`: 348 proteins that
*M. tuberculosis* needs in order to grow. To find their relatives we need more than
their names, because a relative in another organism has a different name and a
different identifier. What the two proteins share is their **sequence**, the string of
amino acids they are made of, so that is what we compare.

First the packages, and the four decisions this notebook makes, written as constants so they are easy to find and easy to change.

In [ ]:
import numpy as np
import pandas as pd
import stylia
from scripts import chembl

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

PCHEMBL_MIN = 5    # a compound counts if it is at least this potent (pChEMBL 5 = 10 uM)
CLOSE = 0.80       # a close homolog is at least 80% identical to our target
DISTANT = 0.40     # a distant homolog is at least 40% identical
MIN_COVERAGE = 0.5  # and the alignment has to cover at least half of our target
print(f"pChEMBL >= {PCHEMBL_MIN}, close >= {CLOSE:.0%}, distant >= {DISTANT:.0%}")

Read the shortlist. It has one row per protein, most vulnerable first.

In [ ]:
targets = pd.read_csv("data/mtb_selected_targets.csv")
print(f"{len(targets)} targets")
targets[["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi"]].head()

The sequences come from UniProt, the main public protein database. We download
the whole *M. tuberculosis* H37Rv reference proteome (`UP000001584`), the same file the
embeddings notebook uses, and read it into a table. A FASTA file writes each protein as
a header line starting with `>` followed by its sequence.

In [ ]:
import gzip
import urllib.request
from pathlib import Path

fasta_path = Path("data/downloads/mtb_proteome.fasta.gz")
fasta_path.parent.mkdir(parents=True, exist_ok=True)
if not fasta_path.exists():
    url = "https://rest.uniprot.org/uniprotkb/stream?query=proteome:UP000001584&format=fasta&compressed=true"
    urllib.request.urlretrieve(url, fasta_path)
with gzip.open(fasta_path, "rt") as f:
    blocks = ("\n" + f.read()).split("\n>")[1:]
proteome = pd.DataFrame([{"uniprot_ac": b.splitlines()[0].split("|")[1],
                          "sequence": "".join(b.splitlines()[1:])} for b in blocks])
print(f"{len(proteome)} proteins in the proteome")

Now we attach a sequence to every target. A few targets are not in the reference proteome, so we ask UniProt for those one by one and check that none is left without a sequence.

In [ ]:
targets = targets.merge(proteome, on="uniprot_ac", how="left")
missing = targets.loc[targets["sequence"].isna(), "uniprot_ac"]
if len(missing):
    extra = chembl.uniprot_sequences(missing).set_index("uniprot_ac")["sequence"]
    targets["sequence"] = targets["sequence"].fillna(targets["uniprot_ac"].map(extra))
assert targets["sequence"].notna().all(), "some targets have no sequence"
print(f"{len(missing)} sequences fetched from UniProt, "
      f"median protein length {int(targets['sequence'].str.len().median())} amino acids")

## 2. What ChEMBL knows about one target

**ChEMBL** collects, from the published literature, molecules that were tested against
proteins and how well they worked. Potency is recorded as a **pChEMBL value**: the
negative logarithm of the concentration needed to get an effect. A pChEMBL value of 5
means 10 micromolar, 6 means 1 micromolar, and 9 means 1 nanomolar, so a bigger number
is a better compound. We count a compound when it reaches at least 5, which is the usual
line between "this molecule does something" and "this is noise".

ChEMBL grows with every release, so a count only means something next to the release it
came from. The cell below prints the release we are using.

In [ ]:
release = chembl.chembl_version()
print(release)

Let's ask about InhA, the protein that isoniazid, one of the first-line TB
drugs, attacks. `targets_for_accession` looks up the protein by its UniProt accession
and returns the ChEMBL entries for it.

In [ ]:
chembl.targets_for_accession("P9WGR1")

InhA is the single-protein target `CHEMBL1849`. Now the question this whole
notebook is about: how many different compounds have been tested on it and reached
pChEMBL 5? This asks ChEMBL directly, over the internet, and takes a few seconds.

In [ ]:
inha_compounds = chembl.live_compounds("CHEMBL1849", PCHEMBL_MIN)
print(f"{len(inha_compounds)} unique compounds with pChEMBL >= {PCHEMBL_MIN} on InhA")
print(sorted(inha_compounds)[:5])

> **Note:** the same compound is often measured many times, in different papers
> and different assays. We keep each compound once: what we want to know is how many
> molecules exist, not how many measurements.

## 3. Every protein target in ChEMBL

Doing that by hand for 348 proteins would be slow, and it would only find the proteins
ChEMBL has studied under the very same accession. Most of our targets have never been
studied, but their relatives in other organisms often have. A series of compounds made
against the *E. coli* version of an enzyme is a real starting point for the
*M. tuberculosis* version, so we let relatives from **any organism** count.

To find them we need every protein ChEMBL knows. ChEMBL also has cell lines, tissues
and protein complexes as targets; only single proteins can be compared by sequence.

The cell below downloads the list of single-protein targets, and shows which organisms they come from.

In [ ]:
chembl_targets = chembl.single_protein_targets()
print(f"{len(chembl_targets)} single-protein targets in {release}")
chembl_targets["organism"].value_counts().head()

Each of those proteins also has a sequence in ChEMBL. Together they are the database we will search. Downloading them takes a few minutes the first time and is then kept on disk.

In [ ]:
chembl_seqs = chembl.target_sequences(set(chembl_targets["accession"]))
print(f"{len(chembl_seqs)} sequences, "
      f"{chembl_seqs['organism'].nunique()} organisms")
chembl_seqs.head(3)

## 4. Find the relatives of every target

Comparing 348 proteins against 11,000 others is done in two steps, because the two
things we want, speed and a trustworthy number, pull in opposite directions.

1. **A fast search** with `phmmer`, a standard tool that compares one protein against a
   whole database in seconds. It gives us the candidates: the proteins that look
   related at all.
2. **A careful alignment** of each candidate, end to end. This is where the number
   comes from: **identity** is the fraction of positions where the two proteins have
   the same amino acid, counted over the whole alignment, and **coverage** is the part
   of our protein that the alignment actually matches up.

The second step matters more than it sounds. A search tool reports the best-matching
*region*, so a small piece shared between two otherwise unrelated proteins can look 85%
identical. Requiring the alignment to cover at least half of our protein
(`MIN_COVERAGE`) is what keeps those out.

Before trusting the measurement, let's check it on pairs whose answer is known: InhA against its *E. coli* counterpart FabI (related, but distantly), the two versions of DNA gyrase B (clearly related), and InhA against itself.

In [ ]:
known = chembl.uniprot_sequences(["P9WGR1", "P0AEK4", "P9WG45", "P0AES6"])
seq = dict(zip(known["uniprot_ac"], known["sequence"]))
for a, b, label in [("P9WGR1", "P0AEK4", "Mtb InhA vs E. coli FabI"),
                    ("P9WG45", "P0AES6", "Mtb GyrB vs E. coli GyrB"),
                    ("P9WGR1", "P9WGR1", "Mtb InhA vs itself")]:
    scores = chembl.global_identity(seq[a], seq[b])
    print(f"{label:30s} identity {scores['identity']:.0%}, "
          f"coverage {scores['query_coverage']:.0%}")

Those are the numbers we expect. The two gyrase B proteins do the same job in two
bacteria and are recognisably related; InhA and FabI also do the same chemistry, but
have drifted further apart; and a protein is of course identical to itself. Notice how
far from 100% "clearly related" already is: proteins that share an ancestor and a
function are often only 30 to 50% identical.

Now the same for all 348 targets at once: the search first, then one alignment per
candidate. Together this takes well under a minute, and the result is saved, so running
the notebook again costs nothing.

In [ ]:
homologs = chembl.homolog_table(targets[["uniprot_ac", "sequence"]], chembl_seqs)
print(f"{len(homologs)} candidate pairs, "
      f"{homologs['uniprot_ac'].nunique()} of our targets have at least one")
homologs.sort_values("identity", ascending=False).head(3)

Each point below is one pair: how identical it is (left to right) against how much of our protein the alignment covers (bottom to top). The dashed lines are our three cutoffs.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(homologs["identity"], homologs["query_coverage"],
           color=nc.orange, s=3, alpha=0.3)
for x in (DISTANT, CLOSE):
    ax.axvline(x, color=nc.gray, linestyle="--")
ax.axhline(MIN_COVERAGE, color=nc.gray, linestyle="--")
stylia.label(ax, xlabel="Identity", ylabel="Part of our protein covered",
             title="Every candidate pair")

Two things stand out. Most candidates sit well to the left of the 40% line: the
search finds distant relatives of almost everything, and most of them are too far away
to tell us anything about chemistry. And almost every pair covers most of our protein,
so the coverage rule turns out to remove very little here. It stays in because the pairs
it would remove, high identity over a small piece of the protein, are exactly the ones
that would fool us.

In [ ]:
enough = homologs["query_coverage"] >= MIN_COVERAGE
print(f"{enough.sum()} pairs of {len(homologs)} cover at least "
      f"{MIN_COVERAGE:.0%} of our protein")

## 5. Sort the matches into three tiers

Every ChEMBL protein we found now goes into one of three tiers:

- **exact**: it is our protein. Either ChEMBL filed it under the same UniProt
  accession, or its sequence is identical to ours. UniProt renumbered most
  *M. tuberculosis* proteins some years ago (`P0A5Y6` became `P9WGR1`), and older
  ChEMBL entries still carry the old number, so we accept those too.
- **close** homolog: at least `CLOSE` identical to our protein.
- **distant** homolog: at least `DISTANT` identical.

The two homolog tiers also have to pass the coverage rule from the last section. The
exact tier does not need it: if ChEMBL says it is the same protein, a short entry is
still that protein.

First the exact ones, which need no sequence comparison at all: we look for our accessions, old numbers included, among the ChEMBL targets.

In [ ]:
alias = chembl.secondary_accessions(targets["uniprot_ac"])
ours = chembl_targets.copy()
ours["uniprot_ac"] = ours["accession"].map(alias).fillna(ours["accession"])
exact = ours[ours["uniprot_ac"].isin(targets["uniprot_ac"])].assign(tier="exact")
print(f"{exact['uniprot_ac'].nunique()} of our {len(targets)} targets are in ChEMBL "
      f"under their own accession")

Now the relatives. `np.select` reads like the list above: the first condition that is true decides, and a sequence identical to ours counts as the protein itself.

In [ ]:
covered = homologs[enough].merge(chembl_targets, on="accession")
covered["tier"] = np.select(
    [covered["identity"] == 1, covered["identity"] >= CLOSE, covered["identity"] >= DISTANT],
    ["exact", "close", "distant"], default="too far")
covered = covered[covered["tier"] != "too far"]
covered["tier"].value_counts()

The two sets go together into one table, one row per (target of ours, ChEMBL protein). Where a protein appears in both, the exact tier wins.

In [ ]:
matches = pd.concat([exact, covered], ignore_index=True)
matches = matches.drop_duplicates(["uniprot_ac", "target_chembl_id"], keep="first")
matches = matches.merge(targets[["uniprot_ac", "gene_name"]], on="uniprot_ac")
print(f"{len(matches)} matches for {matches['uniprot_ac'].nunique()} of our targets, "
      f"over {matches['target_chembl_id'].nunique()} ChEMBL proteins")
matches["tier"].value_counts()

What this looks like for one protein: FtsZ, the protein that builds the ring a bacterium divides along. It is studied as an antibacterial target in several species.

In [ ]:
ftsz = targets.loc[targets["gene_name"] == "ftsZ", "uniprot_ac"].iloc[0]
(matches[matches["uniprot_ac"] == ftsz]
 .sort_values("identity", ascending=False, na_position="first")
 [["pref_name", "organism", "identity", "query_coverage", "tier"]].head(8))

> **Exercise:** change `CLOSE` in section 1 from `0.80` to `0.60` and run the notebook again from here. Which proteins gain a close homolog? Would you defend 60% or 80% to the rest of the group?

## 6. Count the unique compounds

Now the compounds. For each of our targets we count the **unique** molecules tested at
pChEMBL 5 or better, three times over, each count including the one before it:

- `n_compounds_exact`: tested on the protein itself.
- `n_compounds_ge80`: that, plus everything tested on its close homologs.
- `n_compounds_ge40`: that, plus everything tested on its distant homologs.

Counting them cumulatively avoids a trap: the same compound is often tested on a
protein *and* on its relative in another organism. By taking the union of the sets of
molecules, rather than adding counts up, each molecule is counted once no matter how
many of the proteins it was tested on.

First we fetch, for every ChEMBL protein that landed in a tier, the compounds tested on it. This is the slowest cell in the notebook, a couple of minutes, and what it downloads is kept on disk, so running it again is instant.

In [ ]:
matched_ids = matches["target_chembl_id"].unique()
pairs = chembl.precedent_pairs(matched_ids, PCHEMBL_MIN)
print(f"{len(pairs):,} (target, compound) pairs "
      f"for {len(matched_ids)} ChEMBL proteins")

> **Note:** the counting itself is done on sets of compound identifiers, so a compound tested on five relatives of the same target still counts as one.

The cell below turns those pairs into the three cumulative counts, one row per target.

In [ ]:
compounds = pairs.groupby("target_chembl_id")["molecule_chembl_id"].apply(set)

def unique_compounds(group, tiers):
    """Molecules tested on any ChEMBL protein of this target in these tiers."""
    ids = group.loc[group["tier"].isin(tiers), "target_chembl_id"]
    sets = [compounds[i] for i in ids if i in compounds.index]
    return len(set().union(*sets)) if sets else 0

TIERS = {"exact": ["exact"], "ge80": ["exact", "close"],
         "ge40": ["exact", "close", "distant"]}

We apply it target by target, and keep the number of ChEMBL proteins behind each count as well.

In [ ]:
rows = []
for accession, group in matches.groupby("uniprot_ac"):
    row = {"uniprot_ac": accession}
    for name, tiers in TIERS.items():
        row[f"n_targets_{name}"] = group["tier"].isin(tiers).sum()
        row[f"n_compounds_{name}"] = unique_compounds(group, tiers)
    rows.append(row)
counts = pd.DataFrame(rows)
assert (counts["n_compounds_exact"] <= counts["n_compounds_ge80"]).all()
assert (counts["n_compounds_ge80"] <= counts["n_compounds_ge40"]).all()
counts.sort_values("n_compounds_ge40", ascending=False).head()

A good check: the count for InhA should be the number we got straight from ChEMBL in section 2.

In [ ]:
inha_row = counts[counts["uniprot_ac"] == "P9WGR1"].iloc[0]
print(f"InhA: {inha_row['n_compounds_exact']} compounds here, "
      f"{len(inha_compounds)} from the live query in section 2")

## 7. The precedent table

Finally we put the counts next to the targets, add the closest relative of each one as
a name the group can recognise, and label every target with what kind of precedent it
has: its own chemistry, only a close homolog's, only a distant one's, or none at
all.

The closest relative is the best-matching ChEMBL protein that is not our own protein.

In [ ]:
others = matches[matches["tier"] != "exact"].sort_values("identity", ascending=False)
best = (others.drop_duplicates("uniprot_ac")
        .set_index("uniprot_ac")[["target_chembl_id", "pref_name", "organism", "identity"]])
best.columns = ["best_chembl_target_id", "best_chembl_target_name",
                "best_chembl_organism", "best_identity"]
table = targets.drop(columns="sequence").merge(counts, on="uniprot_ac", how="left")
table = table.merge(best, on="uniprot_ac", how="left")
table[[c for c in table.columns if c.startswith("n_")]] = (
    table[[c for c in table.columns if c.startswith("n_")]].fillna(0).astype(int))
print(f"{len(table)} targets, {table['best_identity'].notna().sum()} with a relative")

The label itself follows from the counts: a target either has ChEMBL compounds of its own, or only through a close homolog, or only through a distant one, or none at all.

In [ ]:
table["precedent_class"] = np.select(
    [table["n_compounds_exact"] > 0,
     table["n_compounds_ge80"] > 0,
     table["n_compounds_ge40"] > 0],
    ["own chemistry", "close homolog only", "distant homolog only"], default="none")
table["precedent_class"].value_counts()

The plot below is the one to take back to the group. Each point is a target: how
much chemistry exists for it, counting its relatives as well (left to right, on a
logarithmic scale because the numbers run from zero to over a thousand), against how
badly the bacterium needs it (bottom to top, the vulnerability index, where more
negative means more vulnerable). The interesting corner is the bottom left: proteins the
bacterium cannot do without, that nobody has made compounds for.

In [ ]:
table["precedent"] = np.log10(1 + table["n_compounds_ge40"])
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(table["precedent"], table["vi"], color=nc.orange)
for gene in ["inhA", "folA", "ftsZ", "guaB", "dnaK"]:
    row = table[table["gene_name"] == gene]
    if len(row):
        ax.annotate(gene, (row["precedent"].iloc[0], row["vi"].iloc[0]), fontsize=6)
stylia.label(ax, xlabel="log10(1 + unique compounds, target and relatives)",
             ylabel="Vulnerability index", title="Precedent against vulnerability")

In Colab the cell below also downloads the table to your computer. Upload
`mtb_targets_chembl_precedent.csv` to the group's Drive folder
**Projects/OrangeTeam/Data** so everyone works from the same list.

> **Note:** If nothing downloads, your browser may have blocked it. Look for a message
> near the address bar, allow downloads from Colab, and run the cell again.

In [ ]:
os.makedirs("outputs", exist_ok=True)
final_path = "outputs/mtb_targets_chembl_precedent.csv"
table.drop(columns="precedent").to_csv(final_path, index=False)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(final_path)
print(f"{len(table)} targets written to {final_path} ({release})")

## Summary

- In `ChEMBL_37`, 41 of our 348 targets are in the database under their own accession.
  The sequence search added 3 close homologs (at least 80% identical) and 78 distant
  ones (40 to 80%), so 67 targets have something to learn from and 117 ChEMBL proteins
  were asked about in total.
- 49 targets have at least one compound tested at pChEMBL 5 or better: 31 through their
  own chemistry, 18 only through a relative. The other 299 have none at all.
- Close homologs are almost non-existent: across all targets they add a single compound
  to the exact counts. Nearly all the borrowed chemistry comes from distant relatives,
  such as the *M. avium* dihydrofolate reductase for `folA`, or the mouse enzymes that
  match `glyA1` and `ahcY`. A relative in a mouse is a warning as much as an
  opportunity: a compound that hits both is unlikely to be a safe antibiotic.
- Numbers like these say how much attention a protein has had, not how druggable it is.
  `dnaB`, with 1,127 compounds, has been screened; the 299 targets with none have
  mostly never been tried.

**Next:** put this table next to the pocket scores from `orange_pocket_detection`. A
target that is vulnerable, has a good pocket and has no chemistry yet is exactly what
this project is looking for.